# E-Commerce Lakehouse: Pipeline Validation

## Validate saved tables and reporting totals

Verify expected dataset sizes, unique table keys, and reconciliation
of Gold reporting totals with delivered orders.

These checks apply to this fixed Olist dataset. Expected row counts
must be reviewed if the source dataset changes.

In [0]:
from pyspark.sql import functions as F

expected_counts = {
    "silver_orders": 99441,
    "silver_order_items": 112650,
    "silver_products": 32951,
    "silver_customers": 99441,
    "silver_payments": 103886,
    "gold_order_summary": 99441
}

for table_name, expected in expected_counts.items():
    actual = spark.table(
        f"workspace.ecommerce.{table_name}"
    ).count()

    if actual != expected:
        raise ValueError(
            f"{table_name}: expected {expected}, found {actual}"
        )

    print(f"PASS — {table_name}: {actual} rows")

table_keys = {
    "silver_orders": ["order_id"],
    "silver_order_items": ["order_id", "order_item_id"],
    "silver_products": ["product_id"],
    "silver_customers": ["customer_id"],
    "silver_payments": ["order_id", "payment_sequential"],
    "gold_order_summary": ["order_id"],
    "gold_monthly_sales": ["purchase_month"],
    "gold_category_sales": ["product_category"],
    "gold_customer_metrics": ["customer_unique_id"],
    "gold_delivery_metrics": ["customer_state"]
}

for table_name, keys in table_keys.items():
    duplicate_exists = (
        spark.table(f"workspace.ecommerce.{table_name}")
        .groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_exists:
        raise ValueError(f"Duplicate key found in {table_name}")

print("PASS — all table keys are unique")

delivered_df = spark.table(
    "workspace.ecommerce.gold_order_summary"
).filter(F.col("order_status") == "delivered")

baseline = delivered_df.agg(
    F.count("*").alias("orders"),
    F.sum("merchandise_value").alias("sales")
).first()

for table_name, order_column, sales_column in [
    ("gold_monthly_sales",
     "delivered_order_count", "merchandise_sales_brl"),
    ("gold_customer_metrics",
     "delivered_order_count", "merchandise_sales_brl")
]:
    totals = spark.table(
        f"workspace.ecommerce.{table_name}"
    ).agg(
        F.sum(order_column).alias("orders"),
        F.sum(sales_column).alias("sales")
    ).first()

    if (
        totals["orders"] != baseline["orders"]
        or totals["sales"] != baseline["sales"]
    ):
        raise ValueError(f"Reconciliation failed: {table_name}")

    print(f"PASS — {table_name} reconciles with delivered orders")

category_sales = spark.table(
    "workspace.ecommerce.gold_category_sales"
).agg(
    F.sum("merchandise_sales_brl").alias("sales")
).first()["sales"]

if category_sales != baseline["sales"]:
    raise ValueError("Category sales do not reconcile")

print("PASS — category sales reconcile")

delivery_orders = spark.table(
    "workspace.ecommerce.gold_delivery_metrics"
).agg(
    F.sum("delivered_order_count").alias("orders")
).first()["orders"]

if delivery_orders != baseline["orders"]:
    raise ValueError("Delivery order counts do not reconcile")

print("PASS — delivery order counts reconcile")
print("All checks in this validation suite passed.")

PASS — silver_orders: 99441 rows
PASS — silver_order_items: 112650 rows
PASS — silver_products: 32951 rows
PASS — silver_customers: 99441 rows
PASS — silver_payments: 103886 rows
PASS — gold_order_summary: 99441 rows
PASS — all table keys are unique
PASS — gold_monthly_sales reconciles with delivered orders
PASS — gold_customer_metrics reconciles with delivered orders
PASS — category sales reconcile
PASS — delivery order counts reconcile
All checks in this validation suite passed.
